# v8 GRPO Candidate Policy

Self-contained Kaggle TPU notebook. It downloads `candidates_v7.csv` and the v8 SFT artifact from Hugging Face with `HF_TOKEN`, runs constrained GRPO-style policy improvement on TPU, uploads artifacts to `devaanshpa/orbit-wars-agent/v8/grpo`, pushes JSON checkpoints every 30 epochs, and displays saved metrics/graphs. No companion `.py` files are required.

In [ ]:
from getpass import getpass
import os
from pathlib import Path

os.environ.setdefault("PJRT_DEVICE", "TPU")
os.environ.setdefault("V8_DEVICE", "tpu")

if not os.environ.get("HF_TOKEN"):
    token = getpass("HF_TOKEN: ").strip()
    if not token:
        raise RuntimeError("HF_TOKEN is required to download data and upload v8 artifacts")
    os.environ["HF_TOKEN"] = token

WORK_DIR = Path.cwd()
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
print("PJRT_DEVICE:", os.environ.get("PJRT_DEVICE"))
print("V8_DEVICE:", os.environ.get("V8_DEVICE"))
print("working directory:", WORK_DIR)


In [ ]:
import os

local_csv = os.environ.get("CANDIDATES_CSV", "").strip()
remote_csv = os.environ.get("V8_GRPO_DATA_REMOTE_PATH", "").strip()
local_sft = os.environ.get("V8_SFT_ARTIFACT", "").strip()
remote_sft = os.environ.get("V8_SFT_REMOTE_PATH", "v8/sft/model_weights_v8_sft.json").strip()

if local_csv:
    print("Using explicit local CANDIDATES_CSV:", local_csv)
elif remote_csv:
    print("GRPO will download candidate CSV from Hugging Face:", remote_csv)
else:
    print("GRPO will download the newest data/*/candidates_v7.csv from Hugging Face.")

if local_sft:
    print("Using explicit local V8_SFT_ARTIFACT:", local_sft)
else:
    os.environ["V8_SFT_REMOTE_PATH"] = remote_sft
    print("GRPO will download SFT artifact from Hugging Face:", remote_sft)

print("Upload enabled unless V8_GRPO_UPLOAD=0")
print("Checkpoint interval:", os.environ.get("V8_GRPO_CHECKPOINT_EVERY", "30"), "epochs")


In [ ]:
import importlib.util
import subprocess
import sys

install_missing = [pkg for pkg, module in (("huggingface-hub", "huggingface_hub"), ("matplotlib", "matplotlib")) if importlib.util.find_spec(module) is None]
if install_missing:
    print("Installing missing lightweight packages:", install_missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *install_missing])

if importlib.util.find_spec("torch") is None:
    raise RuntimeError("PyTorch is required in the TPU runtime")
if importlib.util.find_spec("torch_xla") is None:
    raise RuntimeError("torch_xla is required. Switch the Kaggle accelerator to TPU v5e-8 before running v8 on TPU.")

import torch
import torch_xla.core.xla_model as xm
print("torch", torch.__version__)
print("xla device sample:", xm.xla_device())


## Inline Trainer

This cell embeds the v8 GRPO trainer implementation so the notebook can run by itself in Kaggle.

In [ ]:
GRPO_TRAINER_CODE = 'import argparse\nimport csv\nimport json\nimport math\nimport os\nimport random\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\n\nHF_REPO_ID = "devaanshpa/orbit-wars-agent"\nHF_REPO_TYPE = "model"\nSFT_REMOTE_PREFIX = "v8/sft"\nGRPO_REMOTE_PREFIX = "v8/grpo"\n\nMETADATA_COLS = {\n    "label",\n    "selected",\n    "outcome_weight",\n    "game_result",\n    "reward_margin",\n    "agent_reward",\n    "opponent_reward",\n    "selected_heuristic_rank",\n    "counterfactual_positive",\n    "counterfactual_reason",\n    "failure_overcommit",\n    "failure_missed_tactical",\n    "failure_missed_comet",\n    "failure_slow_expansion",\n    "turn_advantage",\n    "future_advantage_delta_5",\n    "future_advantage_delta_15",\n    "future_advantage_delta_30",\n    "future_production_delta_15",\n    "future_planet_delta_15",\n    "game_id",\n    "candidate_id",\n    "version",\n}\n\n\ndef parse_args():\n    parser = argparse.ArgumentParser(\n        description="Run v8 constrained GRPO-style policy improvement over Orbit Wars candidate groups."\n    )\n    parser.add_argument("--csv", default=os.environ.get("CANDIDATES_CSV", ""))\n    parser.add_argument(\n        "--data-remote-path",\n        default=os.environ.get("V8_GRPO_DATA_REMOTE_PATH", ""),\n        help="Optional exact Hugging Face repo path for candidates_v7.csv. If omitted, the newest data/*/candidates_v7.csv is used.",\n    )\n    parser.add_argument(\n        "--prefer-local-data",\n        action="store_true",\n        help="Use a local candidates_v7.csv/candidates_v8.csv before trying Hugging Face. Default is Hugging Face.",\n    )\n    parser.add_argument(\n        "--sft-artifact",\n        default=os.environ.get("V8_SFT_ARTIFACT", ""),\n        help="Optional local SFT JSON override. By default GRPO downloads the SFT artifact from Hugging Face.",\n    )\n    parser.add_argument(\n        "--sft-remote-path",\n        default=os.environ.get("V8_SFT_REMOTE_PATH", f"{SFT_REMOTE_PREFIX}/model_weights_v8_sft.json"),\n        help="Hugging Face repo path for the SFT artifact used by GRPO.",\n    )\n    parser.add_argument(\n        "--prefer-local-sft",\n        action="store_true",\n        help="Use notebooks/v8/exports/sft/model_weights_v8_sft.json before trying Hugging Face.",\n    )\n    parser.add_argument("--export-dir", default="notebooks/v8/exports/grpo")\n    parser.add_argument("--epochs", type=int, default=int(os.environ.get("V8_GRPO_EPOCHS", "120")))\n    parser.add_argument("--batch-groups", type=int, default=int(os.environ.get("V8_GRPO_BATCH_GROUPS", "160")))\n    parser.add_argument("--samples-per-group", type=int, default=int(os.environ.get("V8_GRPO_SAMPLES", "10")))\n    parser.add_argument("--temperature", type=float, default=float(os.environ.get("V8_GRPO_TEMPERATURE", "0.90")))\n    parser.add_argument("--lr", type=float, default=float(os.environ.get("V8_GRPO_LR", "0.00025")))\n    parser.add_argument("--weight-decay", type=float, default=float(os.environ.get("V8_GRPO_WEIGHT_DECAY", "0.00018")))\n    parser.add_argument("--kl-weight", type=float, default=float(os.environ.get("V8_GRPO_KL_WEIGHT", "0.065")))\n    parser.add_argument("--entropy-weight", type=float, default=float(os.environ.get("V8_GRPO_ENTROPY_WEIGHT", "0.012")))\n    parser.add_argument("--supervised-anchor", type=float, default=float(os.environ.get("V8_GRPO_SUPERVISED_ANCHOR", "0.14")))\n    parser.add_argument("--patience", type=int, default=int(os.environ.get("V8_GRPO_PATIENCE", "24")))\n    parser.add_argument("--checkpoint-every", type=int, default=int(os.environ.get("V8_GRPO_CHECKPOINT_EVERY", "30")))\n    parser.add_argument(\n        "--device",\n        choices=("tpu", "cuda", "cpu", "auto"),\n        default=os.environ.get("V8_DEVICE", "tpu"),\n        help="Training device. Defaults to TPU for Kaggle TPU v5e-8 runs.",\n    )\n    parser.add_argument("--seed", type=int, default=1701)\n    parser.add_argument("--upload", action="store_true")\n    parser.add_argument("--hf-repo-id", default=HF_REPO_ID)\n    parser.add_argument("--hf-repo-type", default=HF_REPO_TYPE)\n    return parser.parse_args()\n\n\ndef load_dotenv(path=".env"):\n    env_path = Path(path)\n    if not env_path.exists():\n        return\n    for raw_line in env_path.read_text(encoding="utf-8").splitlines():\n        line = raw_line.strip()\n        if not line or line.startswith("#") or "=" not in line:\n            continue\n        key, value = line.split("=", 1)\n        key = key.strip()\n        value = value.strip().strip("\\"\'")\n        if key and key not in os.environ:\n            os.environ[key] = value\n\n\ndef download_training_csv(remote_path, repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE):\n    load_dotenv()\n    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")\n    if not token:\n        raise RuntimeError("HF_TOKEN is required to download candidates_v7.csv from Hugging Face.")\n    try:\n        from huggingface_hub import HfApi, hf_hub_download\n    except ModuleNotFoundError as exc:\n        raise RuntimeError("Install huggingface_hub to download data: pip install huggingface_hub") from exc\n\n    if not remote_path:\n        api = HfApi(token=token)\n        files = api.list_repo_files(repo_id=repo_id, repo_type=repo_type)\n        remote_csvs = sorted(\n            [\n                name\n                for name in files\n                if name.startswith("data/") and name.endswith("/candidates_v7.csv")\n            ],\n            reverse=True,\n        )\n        if not remote_csvs:\n            raise FileNotFoundError("No data/*/candidates_v7.csv found in Hugging Face repo.")\n        remote_path = remote_csvs[0]\n\n    return Path(\n        hf_hub_download(\n            repo_id=repo_id,\n            repo_type=repo_type,\n            filename=remote_path,\n            token=token,\n        )\n    )\n\n\ndef find_training_csv(csv_arg, remote_path="", prefer_local=False, repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE):\n    if csv_arg:\n        path = Path(csv_arg)\n        if not path.exists():\n            raise FileNotFoundError(f"Training CSV does not exist: {path}")\n        return path\n\n    if not prefer_local:\n        return download_training_csv(remote_path, repo_id=repo_id, repo_type=repo_type)\n\n    root = Path("data")\n    candidates = []\n    if root.exists():\n        candidates.extend(root.glob("*/candidates_v8.csv"))\n        candidates.extend(root.glob("*/candidates_v7.csv"))\n    if candidates:\n        return sorted(candidates, key=lambda path: path.stat().st_mtime, reverse=True)[0]\n\n    return download_training_csv(remote_path, repo_id=repo_id, repo_type=repo_type)\n\n\ndef download_sft_artifact(remote_path, repo_id, repo_type):\n    load_dotenv()\n    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")\n    if not token:\n        raise RuntimeError("HF_TOKEN is required to download the SFT artifact from Hugging Face.")\n    try:\n        from huggingface_hub import hf_hub_download\n    except ModuleNotFoundError as exc:\n        raise RuntimeError("Install huggingface_hub to download SFT artifacts.") from exc\n    return Path(\n        hf_hub_download(\n            repo_id=repo_id,\n            repo_type=repo_type,\n            filename=remote_path,\n            token=token,\n        )\n    )\n\n\ndef find_sft_artifact(path_arg, repo_id, repo_type, remote_path, prefer_local=False):\n    if path_arg:\n        path = Path(path_arg)\n        if not path.exists():\n            raise FileNotFoundError(f"SFT artifact does not exist: {path}")\n        return path\n\n    local = Path("notebooks/v8/exports/sft/model_weights_v8_sft.json")\n    if prefer_local and local.exists():\n        return local\n\n    return download_sft_artifact(remote_path, repo_id, repo_type)\n\n\ndef row_float(row, key, default=0.0):\n    try:\n        return float(row.get(key, default) or default)\n    except (TypeError, ValueError):\n        return float(default)\n\n\ndef read_rows(path):\n    with Path(path).open(newline="", encoding="utf-8") as f:\n        rows = list(csv.DictReader(f))\n    if not rows:\n        raise RuntimeError("Training CSV has no rows.")\n    feature_names = [name for name in rows[0] if name not in METADATA_COLS]\n    x_raw = [[row_float(row, name, 0.0) for name in feature_names] for row in rows]\n    labels = [max(0.0, min(1.0, row_float(row, "label", 0.0))) for row in rows]\n    selected = [row_float(row, "selected", 0.0) >= 0.5 for row in rows]\n    counterfactual = [row_float(row, "counterfactual_positive", 0.0) >= 0.5 for row in rows]\n    return rows, feature_names, x_raw, labels, selected, counterfactual\n\n\ndef split_by_game(rows, seed, validation_fraction=0.2):\n    games = sorted({row.get("game_id", "") for row in rows})\n    rng = random.Random(seed)\n    rng.shuffle(games)\n    valid_game_count = max(1, int(len(games) * validation_fraction)) if len(games) > 1 else 1\n    valid_games = set(games[:valid_game_count])\n    valid_indices = [i for i, row in enumerate(rows) if row.get("game_id", "") in valid_games]\n    valid_set = set(valid_indices)\n    train_indices = [i for i in range(len(rows)) if i not in valid_set] or valid_indices[:]\n    return train_indices, valid_indices, games, valid_games\n\n\ndef normalize_with_artifact(x_raw, feature_names, artifact):\n    member = artifact["members"][0] if artifact.get("members") else artifact\n    means = member.get("mean", {})\n    scales = member.get("scale", {})\n    normalized = []\n    for row in x_raw:\n        values = []\n        for name, value in zip(feature_names, row):\n            scale = float(scales.get(name, 1.0) or 1.0)\n            values.append((value - float(means.get(name, 0.0))) / scale)\n        normalized.append(values)\n    return normalized, means, scales\n\n\ndef build_groups(rows, indices):\n    grouped = defaultdict(list)\n    for raw_index in indices:\n        step = int(row_float(rows[raw_index], "step", 0.0))\n        grouped[(rows[raw_index].get("game_id", ""), step)].append(raw_index)\n    return [items for items in grouped.values() if len(items) >= 2]\n\n\ndef candidate_reward(row, label, selected, counterfactual):\n    reward = (label - 0.5) * 2.0\n    reward += max(-1.5, min(1.5, row_float(row, "future_advantage_delta_15", 0.0) / 80.0))\n    reward += max(-1.0, min(1.0, row_float(row, "future_advantage_delta_30", 0.0) / 120.0)) * 0.55\n    reward += max(-0.8, min(0.8, row_float(row, "future_production_delta_15", 0.0) / 8.0)) * 0.55\n    reward += max(-0.8, min(0.8, row_float(row, "future_planet_delta_15", 0.0) / 3.0)) * 0.45\n    reward += row_float(row, "game_result", 0.0) * 0.22\n    reward += max(-1.0, min(1.0, row_float(row, "reward_margin", 0.0) / 700.0)) * 0.18\n    if selected:\n        reward += 0.18\n    if counterfactual:\n        reward += 0.32\n    reward -= row_float(row, "failure_overcommit", 0.0) * 0.75\n    reward -= row_float(row, "failure_missed_tactical", 0.0) * 0.30\n    reward -= row_float(row, "failure_missed_comet", 0.0) * 0.30\n    reward -= row_float(row, "failure_slow_expansion", 0.0) * 0.25\n    return reward\n\n\ndef target_distribution(rows, rewards, group):\n    values = [math.exp(max(-6.0, min(6.0, rewards[i]))) for i in group]\n    total = sum(values)\n    return [value / total for value in values] if total > 0.0 else [1.0 / len(group)] * len(group)\n\n\ndef grouped_metrics(rows, scores, positive_mask, indices):\n    groups = build_groups(rows, indices)\n    hits = 0\n    total = 0\n    ranks = []\n    for group in groups:\n        positives = [i for i in group if positive_mask[i]]\n        if not positives:\n            continue\n        ordered = sorted(group, key=lambda i: scores[i], reverse=True)\n        total += 1\n        if positive_mask[ordered[0]]:\n            hits += 1\n        ranks.append(min(ordered.index(i) + 1 for i in positives) / max(1, len(ordered)))\n    return {\n        "top1": hits / total if total else 0.0,\n        "rank_fraction": sum(ranks) / len(ranks) if ranks else 1.0,\n        "turns": total,\n    }\n\n\ndef make_model_class(torch, nn):\n    class CandidatePolicy(nn.Module):\n        def __init__(self, feature_count):\n            super().__init__()\n            self.net = nn.Sequential(\n                nn.Linear(feature_count, 128),\n                nn.ReLU(),\n                nn.Linear(128, 64),\n                nn.ReLU(),\n                nn.Linear(64, 32),\n                nn.ReLU(),\n                nn.Linear(32, 1),\n            )\n\n        def forward(self, inputs):\n            return self.net(inputs).view(-1)\n\n    return CandidatePolicy\n\n\ndef load_member_into_model(torch, model, member):\n    linear_layers = [module for module in model.net if module.__class__.__name__ == "Linear"]\n    for layer_module, layer_data in zip(linear_layers, member.get("layers", [])):\n        layer_module.weight.data = torch.tensor(layer_data["weights"], dtype=layer_module.weight.dtype, device=layer_module.weight.device)\n        layer_module.bias.data = torch.tensor(layer_data["bias"], dtype=layer_module.bias.dtype, device=layer_module.bias.device)\n\n\ndef choose_device(torch, args):\n    requested = args.device\n    if requested == "auto":\n        requested = "tpu" if os.environ.get("TPU_NAME") or os.environ.get("PJRT_DEVICE") == "TPU" else "cuda"\n    if requested == "tpu":\n        os.environ.setdefault("PJRT_DEVICE", "TPU")\n        try:\n            import torch_xla.core.xla_model as xm\n        except ModuleNotFoundError as exc:\n            raise RuntimeError("torch_xla is required for V8_DEVICE=tpu. Use a Kaggle TPU v5e-8 runtime.") from exc\n        return xm.xla_device(), xm\n    if requested == "cuda" and torch.cuda.is_available():\n        return torch.device("cuda"), None\n    return torch.device("cpu"), None\n\n\ndef optimizer_step(optimizer, xm):\n    if xm is None:\n        optimizer.step()\n    else:\n        xm.optimizer_step(optimizer, barrier=False)\n        xm.mark_step()\n\n\ndef layers_from_model(torch, nn, model):\n    linear_layers = [module for module in model.net if isinstance(module, nn.Linear)]\n    layers = []\n    for index, layer in enumerate(linear_layers):\n        layers.append(\n            {\n                "weights": layer.weight.detach().cpu().tolist(),\n                "bias": layer.bias.detach().cpu().tolist(),\n                "activation": "relu" if index < len(linear_layers) - 1 else "linear",\n            }\n        )\n    return layers\n\n\ndef maybe_upload_file(args, path, path_in_repo, commit_message):\n    if not args.upload:\n        return\n    load_dotenv()\n    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")\n    if not token:\n        raise RuntimeError("HF_TOKEN is required for checkpoint upload.")\n    try:\n        from huggingface_hub import HfApi\n    except ModuleNotFoundError as exc:\n        raise RuntimeError("Install huggingface_hub to upload checkpoints: pip install huggingface_hub") from exc\n    api = HfApi(token=token)\n    api.create_repo(repo_id=args.hf_repo_id, repo_type=args.hf_repo_type, exist_ok=True)\n    api.upload_file(\n        path_or_fileobj=str(path),\n        path_in_repo=path_in_repo,\n        repo_id=args.hf_repo_id,\n        repo_type=args.hf_repo_type,\n        commit_message=commit_message,\n    )\n    print(f"Uploaded checkpoint to https://huggingface.co/{args.hf_repo_id}/blob/main/{path_in_repo}", flush=True)\n\n\ndef save_grpo_checkpoint(args, torch, nn, model, epoch, item, history, feature_names, means, scales, sft_path, blend):\n    checkpoint_dir = Path(args.export_dir) / "checkpoints"\n    checkpoint_dir.mkdir(parents=True, exist_ok=True)\n    member = {\n        "version": "v8_grpo",\n        "model_type": "mlp_relu_candidate_ranker",\n        "features": feature_names,\n        "mean": means,\n        "scale": scales,\n        "layers": layers_from_model(torch, nn, model),\n        "activation": "relu",\n        "score_scale": 230.0,\n    }\n    payload = {\n        "version": "v8_grpo_checkpoint",\n        "created_at": int(time.time()),\n        "epoch": epoch,\n        "latest_metrics": item,\n        "history": history,\n        "source_sft_artifact": str(sft_path),\n        "model_type": "ensemble_mlp_relu_candidate_ranker",\n        "members": [member],\n        "blend": blend,\n    }\n    path = checkpoint_dir / f"grpo_epoch_{epoch:03d}.json"\n    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")\n    print(f"Saved v8 GRPO checkpoint: {path}", flush=True)\n    maybe_upload_file(\n        args,\n        path,\n        f"{GRPO_REMOTE_PREFIX}/checkpoints/{path.name}",\n        f"Upload v8 GRPO checkpoint epoch {epoch}",\n    )\n\n\ndef sigmoid_prob(value):\n    value = max(-50.0, min(50.0, value))\n    return 1.0 / (1.0 + math.exp(-value))\n\n\ndef train(args):\n    try:\n        import torch\n        from torch import nn\n        import torch.nn.functional as functional\n    except ModuleNotFoundError as exc:\n        raise RuntimeError("PyTorch is required for v8 GRPO training. Install torch or use Kaggle/Colab.") from exc\n\n    random.seed(args.seed)\n    torch.manual_seed(args.seed)\n    data_path = find_training_csv(\n        args.csv,\n        remote_path=args.data_remote_path,\n        prefer_local=args.prefer_local_data,\n        repo_id=args.hf_repo_id,\n        repo_type=args.hf_repo_type,\n    )\n    sft_path = find_sft_artifact(\n        args.sft_artifact,\n        args.hf_repo_id,\n        args.hf_repo_type,\n        args.sft_remote_path,\n        prefer_local=args.prefer_local_sft,\n    )\n    sft_artifact = json.loads(Path(sft_path).read_text(encoding="utf-8"))\n    rows, feature_names, x_raw, labels, selected, counterfactual = read_rows(data_path)\n    all_x, means, scales = normalize_with_artifact(x_raw, feature_names, sft_artifact)\n    train_indices, valid_indices, games, valid_games = split_by_game(rows, args.seed)\n    train_groups = build_groups(rows, train_indices)\n    valid_groups = build_groups(rows, valid_indices)\n    rewards = [\n        candidate_reward(row, labels[i], selected[i], counterfactual[i])\n        for i, row in enumerate(rows)\n    ]\n    positive_mask = [selected[i] or counterfactual[i] or labels[i] >= 0.55 for i in range(len(rows))]\n    device, xm = choose_device(torch, args)\n    CandidatePolicy = make_model_class(torch, nn)\n    base_members = sft_artifact.get("members", []) or [sft_artifact]\n    model = CandidatePolicy(len(feature_names)).to(device)\n    ref_model = CandidatePolicy(len(feature_names)).to(device)\n    load_member_into_model(torch, model, base_members[0])\n    load_member_into_model(torch, ref_model, base_members[0])\n    ref_model.eval()\n    for param in ref_model.parameters():\n        param.requires_grad_(False)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, args.epochs))\n    x_all = torch.tensor(all_x, dtype=torch.float32, device=device)\n\n    print(\n        json.dumps(\n            {\n                "csv": str(data_path),\n                "data_remote_path": args.data_remote_path or "latest data/*/candidates_v7.csv",\n                "data_source": "local_override" if args.csv else ("local_preferred" if args.prefer_local_data else "hugging_face"),\n                "sft_artifact": str(sft_path),\n                "sft_remote_path": args.sft_remote_path,\n                "sft_source": "local_override" if args.sft_artifact else ("local_preferred" if args.prefer_local_sft else "hugging_face"),\n                "rows": len(rows),\n                "features": len(feature_names),\n                "games": len(games),\n                "validation_games": len(valid_games),\n                "train_groups": len(train_groups),\n                "validation_groups": len(valid_groups),\n                "samples_per_group": args.samples_per_group,\n                "device": str(device),\n                "checkpoint_every": args.checkpoint_every,\n                "remote_upload_path": GRPO_REMOTE_PREFIX,\n            },\n            indent=2,\n        ),\n        flush=True,\n    )\n\n    best_state = None\n    best_objective = float("inf")\n    stale = 0\n    history = []\n    checkpoint_blend = max(0.18, min(0.58, float(sft_artifact.get("blend", 0.32)) + 0.08))\n    for epoch in range(1, args.epochs + 1):\n        model.train()\n        groups = train_groups[:]\n        random.shuffle(groups)\n        total_loss = 0.0\n        total_policy = 0.0\n        total_kl = 0.0\n        total_entropy = 0.0\n        batches = 0\n        for offset in range(0, len(groups), args.batch_groups):\n            batch_groups = groups[offset : offset + args.batch_groups]\n            optimizer.zero_grad(set_to_none=True)\n            loss_acc = None\n            policy_acc = 0.0\n            kl_acc = 0.0\n            entropy_acc = 0.0\n            for group in batch_groups:\n                indices = torch.tensor(group, dtype=torch.long, device=device)\n                logits = model(x_all[indices]) / max(0.20, args.temperature)\n                with torch.no_grad():\n                    ref_logits = ref_model(x_all[indices]) / max(0.20, args.temperature)\n                probs = torch.softmax(logits, dim=0)\n                ref_probs = torch.softmax(ref_logits, dim=0)\n                log_probs = torch.log(probs.clamp_min(1e-8))\n                group_rewards = torch.tensor([rewards[i] for i in group], dtype=torch.float32, device=device)\n                sampled = torch.multinomial(probs.detach(), num_samples=min(args.samples_per_group, len(group)), replacement=True)\n                sampled_rewards = group_rewards[sampled]\n                if sampled_rewards.numel() > 1:\n                    advantages = (sampled_rewards - sampled_rewards.mean()) / sampled_rewards.std(unbiased=False).clamp_min(1e-4)\n                else:\n                    advantages = sampled_rewards - sampled_rewards.mean()\n                policy_loss = -(advantages.detach() * log_probs[sampled]).mean()\n                kl = (probs * (torch.log(probs.clamp_min(1e-8)) - torch.log(ref_probs.clamp_min(1e-8)))).sum()\n                entropy = -(probs * torch.log(probs.clamp_min(1e-8))).sum()\n                target = torch.tensor(target_distribution(rows, rewards, group), dtype=torch.float32, device=device)\n                anchor = -(target * functional.log_softmax(logits, dim=0)).sum()\n                loss = policy_loss + args.kl_weight * kl - args.entropy_weight * entropy + args.supervised_anchor * anchor\n                loss_acc = loss if loss_acc is None else loss_acc + loss\n                policy_acc += float(policy_loss.detach().cpu())\n                kl_acc += float(kl.detach().cpu())\n                entropy_acc += float(entropy.detach().cpu())\n            if loss_acc is None:\n                continue\n            loss_acc = loss_acc / max(1, len(batch_groups))\n            loss_acc.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.8)\n            optimizer_step(optimizer, xm)\n            total_loss += float(loss_acc.detach().cpu())\n            total_policy += policy_acc / max(1, len(batch_groups))\n            total_kl += kl_acc / max(1, len(batch_groups))\n            total_entropy += entropy_acc / max(1, len(batch_groups))\n            batches += 1\n        scheduler.step()\n\n        if epoch >= 1:\n            model.eval()\n            with torch.no_grad():\n                logits_all = model(x_all).detach().cpu().tolist()\n            scores = [sigmoid_prob(value) for value in logits_all]\n            valid_metrics = grouped_metrics(rows, scores, positive_mask, valid_indices)\n            objective = (1.0 - valid_metrics["top1"]) + 0.35 * valid_metrics["rank_fraction"] + 0.02 * (total_kl / max(1, batches))\n            item = {\n                "epoch": epoch,\n                "loss": total_loss / max(1, batches),\n                "policy_loss": total_policy / max(1, batches),\n                "kl": total_kl / max(1, batches),\n                "entropy": total_entropy / max(1, batches),\n                "valid_turn_top1": valid_metrics["top1"],\n                "valid_rank_fraction": valid_metrics["rank_fraction"],\n                "objective": objective,\n                "lr": scheduler.get_last_lr()[0],\n            }\n            history.append(item)\n            print(\n                f"grpo epoch={epoch:03d} loss={item[\'loss\']:.5f} "\n                f"policy={item[\'policy_loss\']:.5f} kl={item[\'kl\']:.5f} entropy={item[\'entropy\']:.5f} "\n                f"valid_top1={item[\'valid_turn_top1\']:.4f} rank={item[\'valid_rank_fraction\']:.4f}",\n                flush=True,\n            )\n            if args.checkpoint_every > 0 and epoch % args.checkpoint_every == 0:\n                save_grpo_checkpoint(\n                    args,\n                    torch,\n                    nn,\n                    model,\n                    epoch,\n                    item,\n                    history,\n                    feature_names,\n                    means,\n                    scales,\n                    sft_path,\n                    checkpoint_blend,\n                )\n            if objective + 1e-5 < best_objective:\n                best_objective = objective\n                best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}\n                stale = 0\n            else:\n                stale += 1\n                if stale >= args.patience:\n                    print(f"grpo early_stop epoch={epoch} best_objective={best_objective:.5f}", flush=True)\n                    break\n\n    if best_state is not None:\n        model.load_state_dict(best_state)\n    model.eval()\n    with torch.no_grad():\n        logits_all = model(x_all).detach().cpu().tolist()\n    all_probs = [sigmoid_prob(value) for value in logits_all]\n    train_metrics = grouped_metrics(rows, all_probs, positive_mask, train_indices)\n    valid_metrics = grouped_metrics(rows, all_probs, positive_mask, valid_indices)\n\n    score_scale = 230.0\n    blend = checkpoint_blend\n    member = {\n        "version": "v8_grpo",\n        "model_type": "mlp_relu_candidate_ranker",\n        "features": feature_names,\n        "mean": means,\n        "scale": scales,\n        "layers": layers_from_model(torch, nn, model),\n        "activation": "relu",\n        "score_scale": score_scale,\n    }\n    metrics = {\n        "rows": len(rows),\n        "features": len(feature_names),\n        "games": len(games),\n        "train_groups": len(train_groups),\n        "validation_groups": len(valid_groups),\n        "train_turn_top1_positive_rate": train_metrics["top1"],\n        "validation_turn_top1_positive_rate": valid_metrics["top1"],\n        "train_positive_mean_rank_fraction": train_metrics["rank_fraction"],\n        "validation_positive_mean_rank_fraction": valid_metrics["rank_fraction"],\n        "blend": blend,\n        "score_scale": score_scale,\n        "device": str(device),\n        "kl_weight": args.kl_weight,\n        "entropy_weight": args.entropy_weight,\n        "supervised_anchor": args.supervised_anchor,\n    }\n    artifact = {\n        "version": "v8_grpo",\n        "created_at": int(time.time()),\n        "source_csv": str(data_path),\n        "source_sft_artifact": str(sft_path),\n        "model_type": "ensemble_mlp_relu_candidate_ranker",\n        "members": [member],\n        "blend": blend,\n        "metrics": metrics,\n    }\n\n    export_dir = Path(args.export_dir)\n    graph_dir = export_dir / "graphs"\n    export_dir.mkdir(parents=True, exist_ok=True)\n    graph_dir.mkdir(parents=True, exist_ok=True)\n    (export_dir / "model_weights_v8_grpo.json").write_text(json.dumps(artifact, indent=2, sort_keys=True), encoding="utf-8")\n    (export_dir / "metrics_v8_grpo.json").write_text(json.dumps(metrics, indent=2, sort_keys=True), encoding="utf-8")\n    (export_dir / "training_history_v8_grpo.json").write_text(json.dumps(history, indent=2, sort_keys=True), encoding="utf-8")\n    with (export_dir / "predictions_v8_grpo.csv").open("w", newline="", encoding="utf-8") as f:\n        writer = csv.writer(f)\n        writer.writerow(["row_index", "reward", "label", "prediction", "selected", "counterfactual_positive", "split"])\n        valid_set = set(valid_indices)\n        for i, pred in enumerate(all_probs):\n            writer.writerow([\n                i,\n                rewards[i],\n                labels[i],\n                pred,\n                float(selected[i]),\n                float(counterfactual[i]),\n                "validation" if i in valid_set else "train",\n            ])\n\n    try:\n        import matplotlib.pyplot as plt\n\n        epochs = [item["epoch"] for item in history]\n        plt.figure(figsize=(7, 4))\n        plt.plot(epochs, [item["valid_turn_top1"] for item in history], label="validation top1")\n        plt.plot(epochs, [item["kl"] for item in history], label="KL")\n        plt.plot(epochs, [item["entropy"] for item in history], label="entropy")\n        plt.xlabel("epoch")\n        plt.title("v8 GRPO diagnostics")\n        plt.legend()\n        plt.tight_layout()\n        plt.savefig(graph_dir / "grpo_diagnostics_v8.png", dpi=150)\n        plt.close()\n    except ModuleNotFoundError:\n        print("matplotlib is not installed; skipped graph generation.", flush=True)\n\n    print(json.dumps(metrics, indent=2, sort_keys=True), flush=True)\n    print(f"Saved v8 GRPO artifact: {export_dir / \'model_weights_v8_grpo.json\'}", flush=True)\n\n    if args.upload:\n        load_dotenv()\n        token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")\n        if not token:\n            raise RuntimeError("HF_TOKEN is required for --upload.")\n        try:\n            from huggingface_hub import HfApi\n        except ModuleNotFoundError as exc:\n            raise RuntimeError("Install huggingface_hub to upload: pip install huggingface_hub") from exc\n        api = HfApi(token=token)\n        api.create_repo(repo_id=args.hf_repo_id, repo_type=args.hf_repo_type, exist_ok=True)\n        api.upload_folder(\n            folder_path=str(export_dir),\n            repo_id=args.hf_repo_id,\n            repo_type=args.hf_repo_type,\n            path_in_repo=GRPO_REMOTE_PREFIX,\n            commit_message="Upload v8 GRPO Orbit Wars policy artifacts and graphs",\n        )\n        print(f"Uploaded {export_dir} to https://huggingface.co/{args.hf_repo_id}/tree/main/{GRPO_REMOTE_PREFIX}", flush=True)\n\n\ndef main():\n    train(parse_args())\n\n\nif __name__ == "__main__":\n    main()\n'

grpo_trainer = {"__name__": "inline_v8_grpo"}
exec(GRPO_TRAINER_CODE, grpo_trainer)
print("Loaded inline v8 GRPO trainer")


In [ ]:
from pathlib import Path
from types import SimpleNamespace
import json
import os

export_dir = Path(os.environ.get("V8_GRPO_EXPORT_DIR", str(Path.cwd() / "v8_exports" / "grpo")))
args = SimpleNamespace(
    csv=os.environ.get("CANDIDATES_CSV", ""),
    data_remote_path=os.environ.get("V8_GRPO_DATA_REMOTE_PATH", ""),
    prefer_local_data=os.environ.get("V8_GRPO_PREFER_LOCAL_DATA", "0") == "1",
    sft_artifact=os.environ.get("V8_SFT_ARTIFACT", ""),
    sft_remote_path=os.environ.get("V8_SFT_REMOTE_PATH", "v8/sft/model_weights_v8_sft.json"),
    prefer_local_sft=os.environ.get("V8_GRPO_PREFER_LOCAL_SFT", "0") == "1",
    export_dir=str(export_dir),
    epochs=int(os.environ.get("V8_GRPO_EPOCHS", "120")),
    batch_groups=int(os.environ.get("V8_GRPO_BATCH_GROUPS", "160")),
    samples_per_group=int(os.environ.get("V8_GRPO_SAMPLES", "10")),
    temperature=float(os.environ.get("V8_GRPO_TEMPERATURE", "0.90")),
    lr=float(os.environ.get("V8_GRPO_LR", "0.00025")),
    weight_decay=float(os.environ.get("V8_GRPO_WEIGHT_DECAY", "0.00018")),
    kl_weight=float(os.environ.get("V8_GRPO_KL_WEIGHT", "0.065")),
    entropy_weight=float(os.environ.get("V8_GRPO_ENTROPY_WEIGHT", "0.012")),
    supervised_anchor=float(os.environ.get("V8_GRPO_SUPERVISED_ANCHOR", "0.14")),
    patience=int(os.environ.get("V8_GRPO_PATIENCE", "24")),
    checkpoint_every=int(os.environ.get("V8_GRPO_CHECKPOINT_EVERY", "30")),
    device=os.environ.get("V8_DEVICE", "tpu"),
    seed=int(os.environ.get("V8_GRPO_SEED", "1701")),
    upload=os.environ.get("V8_GRPO_UPLOAD", "1") != "0",
    hf_repo_id=os.environ.get("HF_REPO_ID", "devaanshpa/orbit-wars-agent"),
    hf_repo_type=os.environ.get("HF_REPO_TYPE", "model"),
)

print("GRPO config:", {
    "epochs": args.epochs,
    "batch_groups": args.batch_groups,
    "samples_per_group": args.samples_per_group,
    "temperature": args.temperature,
    "lr": args.lr,
    "weight_decay": args.weight_decay,
    "kl_weight": args.kl_weight,
    "entropy_weight": args.entropy_weight,
    "supervised_anchor": args.supervised_anchor,
    "patience": args.patience,
    "checkpoint_every": args.checkpoint_every,
    "device": args.device,
    "sft_remote_path": args.sft_remote_path,
    "export_dir": str(export_dir),
    "upload": args.upload,
})

grpo_trainer["train"](args)


In [ ]:
from pathlib import Path
import json

export_dir = Path(os.environ.get("V8_GRPO_EXPORT_DIR", str(Path.cwd() / "v8_exports" / "grpo")))
metrics_path = export_dir / "metrics_v8_grpo.json"
if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text(encoding="utf-8")), indent=2, sort_keys=True))
else:
    print("Metrics file not found:", metrics_path)

try:
    from IPython.display import Image, display
    for graph in sorted((export_dir / "graphs").glob("*.png")):
        print(graph)
        display(Image(filename=str(graph)))
except Exception as exc:
    print("Could not display graphs inline:", exc)
    print("Graph files:", sorted((export_dir / "graphs").glob("*.png")))
